In [1]:
import pdfplumber
import pandas as pd
import glob
import re
from tqdm import tqdm
import os

# =====================================
# Folder Path for VRE Reports
# =====================================
pdf_folder = r"C:\Users\rjesh\VRE REPORTS\*.pdf"
output_file = r"C:\Users\rjesh\VRE REPORTS\VRE_Region_Data_All_Days.csv"

# Remove old file if exists
if os.path.exists(output_file):
    os.remove(output_file)

print("Starting VRE Report Extraction...\n")

for file in tqdm(glob.glob(pdf_folder)):
    
    filename = os.path.basename(file)
    
    # Try extracting date from file name (recommended naming)
    date_match = re.search(r'\d{1,2}-[A-Za-z]{3}-\d{2}', filename)
    
    if not date_match:
        date_match = re.search(r'\d{2}\.\d{2}\.\d{2}', filename)
    
    report_date = date_match.group() if date_match else "Unknown"
    
    with pdfplumber.open(file) as pdf:
        
        for page in pdf.pages:
            tables = page.extract_tables()
            
            for table in tables:
                df = pd.DataFrame(table)
                
                # Identify REMC Region Table via key keyword
                if df.astype(str).apply(lambda x: x.str.contains("Installed Capacity").any()).any():
                    
                    df.columns = df.iloc[0]
                    df = df[1:]
                    
                    df = df.dropna(how="all")
                    
                    df["Date"] = report_date
                    
                    df.to_csv(output_file, 
                              mode='a',
                              header=not os.path.exists(output_file),
                              index=False)

print("\n✅ VRE CSV Created Successfully")
print("Saved at:", output_file)

Starting VRE Report Extraction...



100%|████████████████████████████████████████████████████████████████████████████████| 140/140 [15:53<00:00,  6.81s/it]


✅ VRE CSV Created Successfully
Saved at: C:\Users\rjesh\VRE REPORTS\VRE_Region_Data_All_Days.csv
